# Authored conformance experiment
Authorized execution after pre-execution path correction only. No predicate changes. No runtime validation or oracle access.


In [ ]:
import json, hashlib, copy, pathlib, collections, sys
root = pathlib.Path('.')
sys.path.insert(0, str((root / 'artifacts').resolve()))
def digest(path):
    return hashlib.sha256((root / path).read_bytes()).hexdigest()
manifest = json.loads((root / 'artifacts/freeze-manifest.json').read_text())
hashes = {p: digest(p) for p in manifest['sha256']}
assert hashes == manifest['sha256'], 'Freeze hash mismatch: stop before evaluation'
data = json.loads((root / 'artifacts/inputs/cases-v1.json').read_text())
catalog = json.loads((root / 'context/native_catalog.json').read_text())
assert data['catalogSha256'] == hashes['context/native_catalog.json']
assert data['schemaVersion'] == 1 and len(data['cases']) == 24
from obligation_ledger import evaluate, rename_refs


In [ ]:
def scoped_ids(case):
    ids = {n['id'] for n in case['nodes']}
    def walk(x):
        if isinstance(x, dict):
            for k, v in x.items():
                if k in ('node','source','target','required_agent','old','new') and isinstance(v,str):
                    ids.add(v)
                else:
                    walk(v)
        elif isinstance(x,list):
            for v in x: walk(v)
    walk(case)
    return sorted(ids)
def variants(case):
    mapping = {v: 'renamed_node_' + str(i) for i,v in enumerate(scoped_ids(case))}
    yield 'base', copy.deepcopy(case), {}
    yield 'rename', rename_refs(case, mapping), mapping
    perm = copy.deepcopy(case)
    for key in ('nodes','edges','bindings','node_type_assertions'):
        perm[key] = list(reversed(perm.get(key, [])))
    for key in ('route_assertions','capabilities'):
        perm['intent'][key] = list(reversed(perm['intent'].get(key, [])))
    yield 'permutation', perm, {}
    dup = copy.deepcopy(case)
    dup['bindings'] = [copy.deepcopy(b) for b in case['bindings'] for _ in range(2)]
    yield 'duplicate', dup, {}
records = []
for case in data['cases']:
    for variant, state, mapping in variants(case):
        decisions = {m: evaluate(state, catalog, m) for m in ('B0','B1','ledger')}
        records.append(dict(base_case_id=case['case_id'], variant=variant, mapping=mapping, input=state, decisions=decisions))
assert len(records) == 96
counts = {m: dict(collections.Counter(r['decisions'][m]['action'] for r in records)) for m in ('B0','B1','ledger')}
print(json.dumps(counts, indent=2))
print('Base-case action table (not checker verdicts)')
for r in records:
    if r['variant'] == 'base':
        print(r['base_case_id'], *[r['decisions'][m]['action'] for m in ('B0','B1','ledger')])


In [ ]:
output = root / 'artifacts/obligation-results.json'
output.parent.mkdir(parents=True, exist_ok=True)
payload = dict(protocol='artifacts/protocol-v2.md', source_hashes=hashes, freeze_manifest_sha256=digest('artifacts/freeze-manifest.json'), catalog_commit=catalog['langflowCommit'], counts=counts, records=records, limits=['Authored conformance, not learned generation', 'No runtime or provider validation', 'No withheld oracle accessed'])
with output.open('x', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, sort_keys=True)
print(str(output), hashlib.sha256(output.read_bytes()).hexdigest())
